# Latency 압박과 budget 아래의 Multi-agent 팀

우리의 multi-agent 팀은 작업의 일부를 병렬로 수행할 helper agent를 생성할 수 있는 lead agent로 구성됩니다. 팀을 그대로 두면 속도보다 철저함을 최적화합니다. 어느 agent의 context에도 요청자가 얼마나 기다렸는지, 얼마나 더 기다릴 의향이 있는지 나와 있지 않습니다. 이 cookbook에서는 Claude의 시간 인식을 이끌어 내어 작업을 더 빠르게 완수하도록 합니다.

**이 cookbook을 마치면 다음을 할 수 있습니다.**

- prompt caching을 깨뜨리지 않고 팀의 모든 agent에게 동일하게 흐르는 시계를 보여 주기
- latency 압박을 주는 한 문장으로 팀을 더 빠르게 움직이게 하거나 latency budget에 맞춰 속도를 조절하게 하기
- 실행 자체의 log에서 모든 agent가 시계를 봤는지 확인하기

먼저 Messages API만으로 Claude가 얼마나 시간이 지났는지 인식하게 합니다.

1. 작업이 시작될 때 시작하는 **하나의 공유 팀 시계**.
2. **모든 agent가 매 turn 전에 시계를 봅니다.** *어떤* agent에 대해 API를 호출하든 가장 최근 user message(첫 turn에는 작업 프롬프트, 이후에는 tool result)의 끝에 `[elapsed 252s]`라는 짧은 한 줄을 추가합니다. 작업 시작 3분 후 생성된 helper는 첫 turn에 `0s`가 아닌 `[elapsed 182s]`를 봅니다. 시계는 agent가 아닌 팀의 것이기 때문입니다.

그런 다음 다음 두 접근 방식 중 하나로 Claude에 긴박감을 줍니다.

1. **Latency 압박:** 작업 프롬프트와 모든 helper의 작업 지시 마지막에 시간이 중요하다는 한 문장을 넣습니다. 기본 프롬프트와의 차이는 이 한 문장뿐입니다.
2. **Latency budget:** 확장된 시계가 경과 시간과 함께 budget을 표시합니다. 예를 들어 `[elapsed 252s / 600s]`입니다. 별도 프롬프트 텍스트는 없습니다. agent에게 budget에 관해 알려 주는 것은 두 번째 숫자뿐입니다.

이것이 전체 동작 방식이며, 비교할 세 조건이 생깁니다. 시계 없음, 압박을 동반한 시계, budget을 동반한 시계입니다.

> **model에 관한 참고 사항.** Latency incentive는 Claude를 유도하는 새롭게 등장한 방법입니다. 이 notebook은 기본으로 `claude-fable-5-1`을 사용합니다. 이 접근 방식은 Claude Opus 5 및 이전 model에서 충분히 검증되지 않았으므로 `COOKBOOK_MODEL`을 바꾸면 효과가 달라질 수 있습니다.

내장된 작업은 없습니다. 도구, 프롬프트, 질문은 직접 제공합니다. 마지막의 짧은 smoke test는 시계가 모든 agent에 전달되는 것을 보여 줍니다.

orchestration(message hub, 메시지 도구, spawn 도구, agent loop)은 [Async multi-agent orchestration](async_multi_agent_orchestration.ipynb) cookbook을 따릅니다. 구조가 낯설다면 그 cookbook부터 시작하세요.

## 사전 요구 사항

- Python 3.11 이상.
- 필요한 지식: `async`/`await` Python(task, event, timeout)을 편하게 읽을 수 있고, Messages API tool-use loop(model turn, 그 tool call, 되돌려 보내는 tool result)에 익숙해야 합니다. [Async multi-agent orchestration](async_multi_agent_orchestration.ipynb) cookbook은 이 cookbook이 기반으로 삼는 팀의 동작 방식을 다룹니다.
- `ANTHROPIC_API_KEY`에 설정한 Claude API key(또는 `.env` 파일).
- API credit. 각 실행은 lead 하나와 최대 네 helper로 구성되며, 각각 여러 번 호출합니다. 호출은 helper당 15회, lead는 40회로 제한했습니다. notebook을 처음부터 끝까지 실행하면 effort `xhigh`의 `claude-fable-5-1`에서 약 1분간 짧은 팀 실행 한 번이 이루어집니다.
- rate limit의 여유. 최대 다섯 agent가 동시에 API를 호출하며, 낮은 tier에서는 SDK의 자동 retry가 오류 대신 추가 실제 경과 시간으로 나타납니다.

In [1]:
%%capture
%pip install -qU anthropic python-dotenv

In [2]:
import asyncio
import itertools
import os
import re
import time
from collections import Counter, defaultdict

import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.AsyncAnthropic()  # 환경에서 ANTHROPIC_API_KEY를 읽음

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-fable-5-1")  # 모든 agent; 어떤 현재 model이든 가능
EFFORT = os.environ.get("COOKBOOK_EFFORT", "xhigh")  # low | medium | high | xhigh | max
USER_ID = os.environ.get("COOKBOOK_USER_ID")  # 선택 사항: 설정하면 metadata.user_id로 전송
MAX_HELPERS = 4  # 동시에 실행되는 helper 수 제한(비용 보호 장치이며 동작 방식의 일부가 아님)
PROGRESS_EVERY = 30  # 실행이 출력하는 진행 줄 사이의 초 간격; None이면 출력하지 않음

## 팀 시계

stopwatch와 한 줄 renderer입니다. 작업당 시계 객체는 **하나**이며 팀의 모든 agent가 그 참조를 보유합니다. 그래서 늦게 생성된 helper도 자신의 경과 시간이 아닌 팀의 경과 시간을 봅니다. 선택적인 budget도 같은 객체에 있으므로 팀의 것입니다. 600초 budget 중 182초가 지난 시점에 생성된 helper는 새로운 600초가 아니라 `[elapsed 182s / 600s]`를 봅니다.

이 줄은 한 곳인 `clock_line`에서 정수 초 단위로 렌더링하므로, budget이 있으면 두 숫자의 단위가 같고 한 번의 뺄셈으로 남은 시간을 알 수 있습니다. 대괄호는 앞에 있는 프롬프트나 tool result와 이 줄을 구분하기 위한 것입니다.

In [3]:
def clock_line(elapsed: float, budget: float | None = None) -> str:
    """agent에게 보이는 줄: '[elapsed 192s]', 또는 budget이 있으면 '[elapsed 192s / 600s]'."""
    line = f"elapsed {round(elapsed)}s"
    if budget is not None:
        line += f" / {round(budget)}s"
    return f"[{line}]"


class TeamClock:
    def __init__(self, budget: float | None = None):
        self.start = time.monotonic()
        self.budget = budget  # 초 단위; 설정하면 줄에 표시되며 강제하지 않음

    @property
    def elapsed(self) -> float:
        return time.monotonic() - self.start

    def note(self) -> str:
        return clock_line(self.elapsed, self.budget)


def stamp_elapsed(messages: list[dict], clock: TeamClock) -> str:
    """가장 최근 user message의 끝에 시계 줄을 추가하고 그 줄을 반환합니다.

    작업 프롬프트 뒤에는 별도 text block이 되고, tool result 뒤에는 마지막 결과의
    텍스트를 확장합니다. 이전 줄들은 history에 남으므로 agent는 자신의 각 단계에
    얼마나 시간이 걸렸는지 볼 수 있습니다.
    """
    line = clock.note()
    last = messages[-1]
    content = last["content"]
    if isinstance(content, str):  # 작업 프롬프트: 별도 text block으로 줄 추가
        last["content"] = [{"type": "text", "text": content}, {"type": "text", "text": line}]
    elif content and content[-1].get("type") == "tool_result":  # tool result: 마지막 결과 확장
        content[-1]["content"] += f"\n\n{line}"
    else:
        content.append({"type": "text", "text": line})
    return line


print(clock_line(192), "   with a 600-second budget:", clock_line(192, 600))

demo_clock = TeamClock()
demo_clock.start -= 192  # 192초가 지났다고 가정
demo_messages = [{"role": "user", "content": "Find the best settings for our process."}]
stamp_elapsed(demo_messages, demo_clock)
demo_messages  # model이 첫 turn에 받는 것과 정확히 같음

[elapsed 192s]    with a 600-second budget: [elapsed 192s / 600s]


[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Find the best settings for our process.'},
   {'type': 'text', 'text': '[elapsed 192s]'}]}]

## Hub와 조율 도구

**조율 도구**는 orchestration cookbook의 client-side 도구입니다. 모든 agent가 메모리 내 hub를 통해 `send_message` / `wait_for_message`를 호출할 수 있고, lead는 `create_subagents`를 호출할 수 있습니다. helper가 끝나면 그 최종 답변이 lead에게 자동으로 전달됩니다. hub는 실행의 agent별 tool call 횟수와 log도 보관합니다.

In [4]:
def tool(name: str, description: str, properties: dict, required: list[str]) -> dict:
    schema = {"type": "object", "properties": properties, "required": required}
    return {"name": name, "description": description, "input_schema": schema}


SEND_MESSAGE = tool(
    "send_message",
    "Send a message to one or more other agents on your team. It is appended to their next "
    "tool result.",
    {
        "recipient_ids": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "content": {"type": "string"},
    },
    ["recipient_ids", "content"],
)
WAIT_FOR_MESSAGE = tool(
    "wait_for_message",
    "Block until another agent messages you or a helper finishes. Messages also arrive appended "
    "to the result of any other tool call.",
    {},
    [],
)
CREATE_SUBAGENTS = tool(
    "create_subagents",
    "Spawn helper agents that work in parallel with you. Returns immediately with their ids. "
    "Each helper gets base_instruction plus its own entry of per_helper_instructions, has the "
    "same task tools as you, and reports its findings back to you when it finishes.",
    {
        "base_instruction": {"type": "string"},
        "per_helper_instructions": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": MAX_HELPERS,
        },
    },
    ["base_instruction", "per_helper_instructions"],
)


class Hub:
    """한 실행의 공유 상태: agent별 inbox와 event, 실행의 횟수와 log."""

    def __init__(self):
        self.inbox: dict[str, list[dict]] = defaultdict(list)
        self.event: dict[str, asyncio.Event] = defaultdict(asyncio.Event)
        self.status: dict[str, str] = {}
        self.usage: dict[str, Counter] = defaultdict(Counter)  # agent별 작업 도구 호출 횟수
        self.log: list[dict] = []  # 모든 agent의 진행 항목; run_team이 반환
        self._ids = itertools.count(1)

    def register(self, name: str) -> str:
        self.status[name] = "active"
        return name

    def new_helper_name(self) -> str:
        return self.register(f"helper{next(self._ids)}")

    def post(self, sender: str, recipients: list[str], content: str) -> list[str]:
        delivered = []
        for rid in recipients:
            if self.status.get(rid) in ("active", "waiting"):  # 끝난 agent는 아무것도 읽지 않음
                self.inbox[rid].append({"from": sender, "content": content})
                self.event[rid].set()
                delivered.append(rid)
        return delivered

    def drain(self, name: str) -> str:
        msgs, self.inbox[name] = self.inbox[name], []
        self.event[name] = asyncio.Event()
        if not msgs:
            return ""
        body = "\n".join(
            f'<agent-message from="{m["from"]}">\n{m["content"]}\n</agent-message>' for m in msgs
        )
        return f"\n\n[Messages received while you were working:]\n{body}"

## 자신의 작업 도구

직접 채워 넣을 부분입니다. `TASK_TOOLS`는 모든 agent에게 제공되는 도구 정의 목록(동일한 작은 `tool` helper로 작성)이고, `make_task_handlers(hub)`는 도구를 실행하는 함수들을 반환합니다. 도구 이름마다 `async fn(agent, input)` 하나가 있으며 실행마다 새로 생성됩니다. 둘 다 처음에는 비어 있으므로 기본 상태의 팀은 model 자체의 지식을 바탕으로 작업합니다. 주석으로 된 template은 도구 하나의 형태를 보여 줍니다.

> **참고: Client-side 도구는 시계를 최신으로 유지합니다.** harness는 API 호출 사이에만 시계 줄을 추가할 수 있습니다. 서버에서 응답이 sampling되는 동안에는 그곳에서 실행되는 server-side 도구를 포함해 아무것도 삽입할 수 없으므로, 호출이 반환될 때까지 agent는 마지막으로 본 시계 줄을 바탕으로 작업합니다. client-side tool call마다 harness에 제어를 반환하며 다음 요청에는 갱신된 줄이 포함됩니다. agent 작업이 client-side 도구를 많이 거칠수록 시계가 더 최신 상태로 유지됩니다.

In [5]:
TASK_TOOLS: list[dict] = []  # 자신의 도구 정의: 팀의 모든 agent가 받음


def make_task_handlers(hub: Hub) -> dict:
    """한 실행에서 TASK_TOOLS를 구현하는 함수: {tool name: async fn(agent, input) -> str}."""
    return {}


# client-side 도구 하나의 template. `agent`는 호출하는 agent의 이름("lead", "helper2", ...)입니다.
#
# TASK_TOOLS = [
#     tool(
#         "lookup_record",
#         "Look up one record by its key in the records system. Takes a few seconds.",
#         {"key": {"type": "string"}},
#         ["key"],
#     )
# ]
#
#
# def make_task_handlers(hub: Hub) -> dict:
#     async def lookup_record(agent: str, args: dict) -> str:
#         return await records.get(args["key"])  # 자신의 코드를 여기에 작성
#
#     return {"lookup_record": lookup_record}

## Agent loop와 시계를 넣는 위치

하나의 coroutine이 lead든 helper든 모든 agent를 실행합니다. 일반적인 tool-use loop에 두 가지를 추가했고, 둘 다 코드에 표시되어 있습니다.

- **매 API 호출 직전**, `stamp_elapsed`가 가장 최근 user message에 팀 시계 줄을 추가합니다. 첫 turn에는 이 메시지가 작업 프롬프트이므로 agent는 어떤 행동도 하기 전에 경과 시간을 압니다. 이후에는 다른 agent의 메시지 옆, 마지막 tool result 끝에 줄이 들어갑니다. 전송할 때 시간 표시를 추가하므로 helper나 느린 도구를 기다린 시간도 나타납니다.
- **helper가 끝나면**, 최종 텍스트가 lead의 inbox에 게시됩니다. 그 전에 작업 중 도착한 메시지를 읽으므로 lead가 방향을 바꾸라는 지시를 보내도 놓치지 않습니다.

API의 세부 사항 두 가지입니다. 응답 `content`(thinking이 포함될 수 있음)는 반환된 그대로 history에 들어갑니다. client-side `tool_use` block에만 우리가 `tool_result`를 제공해야 합니다. 그리고 최상위 `cache_control`은 [자동 prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching)을 켭니다. 이는 history가 tool result로 채워지는 agent에게 중요합니다. 시계 줄을 맨 끝에 추가하므로 그 앞의 모든 내용은 일반적으로 다음 호출에서도 cache hit가 됩니다.

한 turn의 agent tool call들은 동시에 실행됩니다. loop는 각 agent에게 보여 준 내용도 실행 log에 기록합니다(`[task]`, `[clock]`, `[report]` 항목). 마지막 섹션에서 이를 다시 읽습니다.

In [6]:
TASK_TOOL_NAMES = {t["name"] for t in TASK_TOOLS}


def log(hub: Hub, clock: TeamClock, name: str, text) -> None:
    """실행 hub에 진행 항목 하나를 기록합니다. 아무것도 출력하지 않습니다."""
    text = " ".join(str(text).split())
    hub.log.append({"seconds": round(clock.elapsed), "agent": name, "text": text})


async def call_tool(hub: Hub, clock: TeamClock, name: str, block, dispatch: dict) -> dict:
    """agent의 tool call 하나를 실행하고 해당 tool_result block을 반환합니다."""
    if block.name == "send_message":
        ids = block.input["recipient_ids"]
        sent = hub.post(name, ids, block.input["content"])
        missed = [r for r in ids if r not in sent]
        out = f"delivered to {sent or 'no one'}" + (f"; not running: {missed}" if missed else "")
    elif block.name == "wait_for_message":
        hub.status[name] = "waiting"
        try:
            await asyncio.wait_for(hub.event[name].wait(), timeout=90)
            out = "new messages below"
        except TimeoutError:
            out = "no messages yet (waited 90s)"
        hub.status[name] = "active"
    elif block.name in dispatch:
        if block.name in TASK_TOOL_NAMES:
            hub.usage[name][block.name] += 1
        out = await dispatch[block.name](name, block.input)
    else:
        out = f"error: unknown tool {block.name}"
    log(hub, clock, name, f"-> {block.name}({block.input}) = {out}")
    return {"type": "tool_result", "tool_use_id": block.id, "content": out}


async def run_agent(
    hub: Hub,
    clock: TeamClock,
    name: str,
    *,
    system: str,
    task: str,
    tools: list[dict],
    dispatch: dict,
    report_to: str | None = None,
    show_clock: bool = True,
    max_turns: int = 40,
) -> str:
    """agent 하나를 완료될 때까지 실행하고 최종 텍스트를 반환합니다."""
    messages = [{"role": "user", "content": task}]
    log(hub, clock, name, f"[task] {task}")  # 이 agent에게 요청한 내용으로, 실행 log를 위해 보관
    final = f"[{name} stopped after max_turns={max_turns}]"
    try:
        for _ in range(max_turns):
            if show_clock:
                line = stamp_elapsed(messages, clock)  # <- 추가 1: 매 호출 전의 시계
                log(hub, clock, name, f"[clock] {line}")
            resp = await client.messages.create(
                model=MODEL,
                max_tokens=16000,  # SDK는 큰 non-streaming 요청을 거부하므로 더 큰 요청에는 streaming 사용
                system=system,
                tools=tools,
                messages=messages,
                cache_control={"type": "ephemeral"},  # 자동 prompt caching
                output_config={"effort": EFFORT},
                **({"metadata": {"user_id": USER_ID}} if USER_ID else {}),
            )
            messages.append({"role": "assistant", "content": resp.content})
            texts = [b.text for b in resp.content if b.type == "text" and b.text.strip()]
            for text in texts:
                log(hub, clock, name, text)

            calls = [b for b in resp.content if b.type == "tool_use"]
            if resp.stop_reason != "tool_use" or not calls:  # end_turn(또는 max_tokens): 종료
                # 정상적인 turn 종료 시 helper는 먼저 작업 중 도착한 내용을 읽음,
                # 따라서 lead의 방향 전환 지시나 후속 지시가 유실되지 않고 처리됨.
                mail = hub.drain(name) if report_to and resp.stop_reason == "end_turn" else ""
                if mail:
                    messages.append({"role": "user", "content": mail.lstrip()})
                    continue
                final = texts[-1].strip() if texts else f"[{name} ended with {resp.stop_reason}]"
                if (
                    resp.stop_reason == "max_tokens"
                ):  # 중단됨: 이를 명시하지 않으면 빠른 실행처럼 보임
                    final = f"[{name} hit max_tokens; answer truncated] {final}"
                break

            # 한 turn의 tool call들을 동시에 실행
            results = await asyncio.gather(
                *(call_tool(hub, clock, name, b, dispatch) for b in calls)
            )
            results[-1]["content"] += hub.drain(name)  # inbox를 마지막 결과에 함께 전달
            messages.append({"role": "user", "content": list(results)})
        hub.status[name] = "done"
    except asyncio.CancelledError:
        hub.status[name] = "cancelled"
        raise
    except Exception as e:  # helper의 crash가 팀 전체를 중단시키면 안 됨
        hub.status[name] = "crashed"
        final = f"[{name} crashed: {e!r}]"
        log(hub, clock, name, final)
        if report_to is None:
            raise
    if report_to:  # <- 추가 2: 종료된 helper가 lead에게 자동 보고
        hub.post(name, [report_to], f"(final report)\n{final}")
        log(hub, clock, name, f"[report] delivered to {report_to}")
    return final

## 프롬프트

- `TIME_MATTERS`는 lead의 작업**과** 각 helper의 작업 지시에 한 번씩 추가됩니다. 기본 프롬프트와 압박을 주는 프롬프트의 유일한 차이입니다. 여러 표현을 실험했으며 이 표현이 latency와 품질 사이에 받아들일 만한 균형을 제공한다고 생각합니다.
- budget은 **프롬프트 텍스트를 추가하지 않습니다**. `latency_budget_seconds`를 설정하면 시계 줄에 두 번째 숫자가 붙어 `[elapsed 252s / 600s]`가 되며, agent에게 알려 주는 것은 이것뿐입니다.
- system prompt에는 시간에 관한 내용이 없습니다.

`run_team(question)`은 시계 없이 기본 팀을 실행합니다. `latency_pressure=True`는 시계와 문장을 추가하고, `latency_budget_seconds=N`은 budget이 표시된 시계를 추가합니다. 별개의 조건이므로 둘 다 요청하면 오류가 발생합니다. 실행은 `PROGRESS_EVERY`초마다 진행 줄을 출력하고 답변, 실제 경과 초, agent별 tool call 횟수, 전체 log를 반환합니다.

In [7]:
TIME_MATTERS = (
    "\n\nTime matters here: do not spend time that can be avoided, and the earlier a correct "
    "result is obtained, the better."
)

LEAD_SYSTEM = (
    "You are the lead of a small team. You can work on the task yourself, and you can spawn up to "
    f"{MAX_HELPERS} helper agents with create_subagents to work on parts of it in parallel; each "
    "helper has the same task tools as you and reports back to you when it finishes (use "
    "wait_for_message to collect reports, send_message to redirect a helper). Decompose the task, "
    "delegate the independent parts, check what comes back, and finish with a short final answer "
    "that states the answer on its first line."
)
HELPER_SYSTEM = (
    "You are {name}, a helper on a team led by the agent 'lead'. Do exactly what your brief asks. "
    "When you are done, reply with your findings: the answer first, then the key evidence. That "
    "final reply is delivered to the lead automatically. If the lead messages you a follow-up, "
    "handle it the same way."
)


def total_calls(hub: Hub) -> int:
    return sum(sum(counts.values()) for counts in hub.usage.values())


async def run_team(
    question: str,
    *,
    latency_pressure: bool = False,
    latency_budget_seconds: float | None = None,
) -> dict:
    """세 조건 중 하나를 실행하며 혼합하지 않습니다.

    시계 없음(기본값): 기본 프롬프트를 사용하며 시계 줄 없음.
    압박(latency_pressure=True): 시간이 중요하다는 문장과 시계 줄.
    budget(latency_budget_seconds=N): budget이 표시된 시계 줄이며 문장 없음.
    """
    budget = latency_budget_seconds
    if budget is not None and budget < 1:
        raise ValueError("latency_budget_seconds must be at least 1 (it is in seconds), or None")
    if latency_pressure and budget is not None:
        raise ValueError("pressure and a budget are separate arms: pass one or the other")
    hub, clock = Hub(), TeamClock(budget)  # 팀 전체에 시계 하나와 budget 하나
    hub.register("lead")
    helpers: dict[str, asyncio.Task] = {}
    show_clock = latency_pressure or budget is not None
    # 압박은 각 프롬프트에 한 문장을 추가. budget은 단어를 추가하지 않고 시계 줄만
    # 변경하며, TeamClock에 이미 budget이 포함되어 있음.
    time_matters = TIME_MATTERS if latency_pressure else ""
    task_handlers = make_task_handlers(hub)

    async def create_subagents(agent: str, args: dict) -> str:
        briefs = args["per_helper_instructions"]
        free = MAX_HELPERS - sum(not t.done() for t in helpers.values())
        spawned = []
        for brief in briefs[: max(free, 0)]:
            h = hub.new_helper_name()
            coro = run_agent(
                hub,
                clock,  # 팀 시계: helper의 첫 줄에는 0s가 아니라 팀의 경과 시간을 표시
                h,
                system=HELPER_SYSTEM.format(name=h),
                task=f"{args['base_instruction']}\n\n{brief}".strip() + time_matters,
                tools=[*TASK_TOOLS, SEND_MESSAGE, WAIT_FOR_MESSAGE],
                dispatch=task_handlers,
                report_to="lead",
                show_clock=show_clock,
                max_turns=15,
            )
            helpers[h] = asyncio.create_task(coro)
            spawned.append(h)
        held = len(briefs) - len(spawned)
        note = f" ({held} not spawned: at most {MAX_HELPERS} helpers at once)" if held else ""
        return f"spawned: {spawned}{note}"

    async def report_progress() -> None:
        while PROGRESS_EVERY:  # None 또는 0이면 진행 줄을 끔
            await asyncio.sleep(PROGRESS_EVERY)
            running = sum(s in ("active", "waiting") for s in hub.status.values())
            print(
                f"[{round(clock.elapsed):>5}s] {running} agent(s) running, "
                f"{total_calls(hub)} tool calls"
            )

    progress = asyncio.create_task(report_progress())
    try:
        answer = await run_agent(
            hub,
            clock,
            "lead",
            system=LEAD_SYSTEM,
            task=question + time_matters,
            tools=[*TASK_TOOLS, CREATE_SUBAGENTS, SEND_MESSAGE, WAIT_FOR_MESSAGE],
            dispatch={**task_handlers, "create_subagents": create_subagents},
            show_clock=show_clock,
        )
    finally:  # 답변 도착; 아직 실행 중인 helper는 더 이상 필요하지 않음
        for t in [progress, *helpers.values()]:
            t.cancel()
        await asyncio.gather(progress, *helpers.values(), return_exceptions=True)
    return {
        "answer": answer,
        "seconds": round(clock.elapsed),
        "helpers": len(helpers),
        "budget": budget,
        "tool_calls": total_calls(hub),
        "usage": {agent: dict(counts) for agent, counts in hub.usage.items()},
        "log": hub.log,
    }

## End-to-end test

자신의 도구를 추가하기 전에 연결 상태를 확인하는 짧은 실행입니다. `TASK_TOOLS`가 비어 있으면 팀은 model 자체의 지식으로 작업하며, 아래 질문은 lead가 helper를 사용하도록 하여 관찰할 내용을 만듭니다. 예제 작업이 아니라 확인용입니다. 실행 후 세 개의 작은 deterministic reader가 실행 자체의 log에 기록된 내용을 출력합니다.

- `print_summary`는 답변과 각 agent의 tool call을 출력합니다.
- `show_first_prompts`는 lead와 첫 helper가 받은 첫 메시지와 그 뒤의 시계 줄을 출력합니다. 압박 조건에서는 둘 다 끝에 시간이 중요하다는 문장이 있습니다.
- `show_clock_lines`는 agent별로 보여 준 시계 줄을 출력합니다. helper의 첫 줄은 `0s`가 아닌 팀의 경과 시간입니다.
- `show_timeline`은 lead가 helper를 생성한 시점, 각각이 보고한 시점, 답변이 나온 시점을 출력하며, budget이 있으면 함께 비교합니다.

budget 조건을 확인하려면 주석 처리된 줄로 교체하세요.

In [8]:
QUESTION = (
    "Wiring check. Spawn one helper for each of these four sorting algorithms: quicksort, "
    "mergesort, heapsort and timsort. Have each report the average and worst-case time complexity and whether "
    "the sort is stable. Then put the four reports in one small table, and say on the first line "
    "which of the four Python's built-in sort uses."
)


def print_summary(result: dict) -> None:
    budget = result.get("budget")
    against = f" against a {round(budget)}s budget" if budget else ""
    print("=" * 88)
    print(
        f"ANSWER after {result['seconds']}s wall-clock{against}, "
        f"{result['helpers']} helper(s) spawned:\n{result['answer']}\n"
    )
    for agent, counts in sorted(result["usage"].items(), key=lambda kv: _agent_order(kv[0])):
        print(f"  {agent:<8} " + "  ".join(f"{tool} {n:>3}" for tool, n in sorted(counts.items())))
    print(f"  {'TOTAL':<8} task-tool calls {result['tool_calls']:>3}")


def _agent_order(agent: str) -> tuple:
    return (agent != "lead", int(re.sub(r"\D", "", agent) or 0))


def _tagged(result: dict, tag: str, agent: str | None = None) -> list[dict]:
    log_ = result["log"]
    return [e for e in log_ if e["text"].startswith(tag) and agent in (None, e["agent"])]


def show_first_prompts(result: dict, width: int = 200) -> None:
    """lead와 첫 helper가 받은 첫 메시지와 그 뒤의 시계 줄."""
    for agent in ("lead", "helper1"):
        tasks = _tagged(result, "[task] ", agent)
        if not tasks:
            continue
        text = tasks[0]["text"][len("[task] ") :]
        if len(text) > 2 * width:
            text = f"{text[:width]} [...] {text[-width:]}"
        clocks = _tagged(result, "[clock] ", agent)
        line = clocks[0]["text"][len("[clock] ") :] if clocks else "(no clock line)"
        print(f"{agent}, first message:\n  {text}\n  {line}\n")


def show_clock_lines(result: dict, limit: int = 8) -> None:
    """agent별로 보여 준 시계 줄을 순서대로 표시합니다."""
    seen = defaultdict(list)
    for e in _tagged(result, "[clock] "):
        seen[e["agent"]].append(e["text"][len("[clock] ") :])
    if not seen:
        print("No agent was shown a clock in this run.")
    for agent in sorted(seen, key=_agent_order):
        lines = seen[agent]
        more = f" -> ... ({len(lines) - limit} more)" if len(lines) > limit else ""
        print(f"{agent:<8} {' -> '.join(lines[:limit])}{more}")


def show_timeline(result: dict) -> None:
    """lead가 helper를 생성한 시점, 각각이 보고한 시점, 답변이 도착한 시점."""
    budget = result.get("budget")
    events = [(0, "lead starts" + (f", budget {round(budget)}s" if budget else ""))]
    for e in _tagged(result, "-> create_subagents(", "lead"):
        names = re.findall(r"helper\d+", e["text"].split("= spawned:")[-1])
        if names:
            events.append((e["seconds"], f"lead spawns {', '.join(names)}"))
    for e in _tagged(result, "[report] "):
        events.append((e["seconds"], f"{e['agent']} reports"))
    verdict = ""
    if budget:
        gap = round(budget) - result["seconds"]
        verdict = f" ({gap}s inside the budget)" if gap >= 0 else f" ({-gap}s past the budget)"
    events.append((result["seconds"], f"lead answers{verdict}"))
    for seconds, what in sorted(events, key=lambda x: x[0]):
        print(f"{seconds:>4}s  {what}")

In [9]:
# Smoke test: 압박 조건으로 한 번 실행한 후 agent에게 보여 준 내용을 다시 읽음
result = await run_team(QUESTION, latency_pressure=True)
# result = await run_team(QUESTION, latency_budget_seconds=60)  # 대신 budget 조건 사용
print_summary(result)
print("\n--- what the agents were told ---")
show_first_prompts(result)
print("--- the clock lines each agent saw ---")
show_clock_lines(result)
print("\n--- timeline ---")
show_timeline(result)

ANSWER after 9s wall-clock, 4 helper(s) spawned:
**Timsort** — Python's built-in `sorted()` / `list.sort()` uses it.

| Algorithm | Average | Worst | Stable |
|-----------|---------|-------|--------|
| Quicksort | O(n log n) | O(n²) | No |
| Mergesort | O(n log n) | O(n log n) | Yes |
| Heapsort  | O(n log n) | O(n log n) | No |
| Timsort   | O(n log n) | O(n log n) | Yes |

All four helpers reported back; results match standard references.

  TOTAL    task-tool calls   0

--- what the agents were told ---
lead, first message:
  Wiring check. Spawn one helper for each of these four sorting algorithms: quicksort, mergesort, heapsort and timsort. Have each report the average and worst-case time complexity and whether the sort i [...] mall table, and say on the first line which of the four Python's built-in sort uses. Time matters here: do not spend time that can be avoided, and the earlier a correct result is obtained, the better.
  [elapsed 0s]

helper1, first message:
  Report, in one 

## 결론

lead와 helper로 이루어진 팀에 하나의 공유 시계를 제공하고 모든 agent에게 매 turn 전에 보여 주었습니다. 그다음 latency 압박을 주는 한 문장이나 시계 줄의 budget이 팀의 속도 조절 방식을 바꿉니다.

- **압박을 사용하세요**, 단순히 빠를수록 좋고 구체적인 숫자가 없을 때입니다. 조정이 필요하지 않습니다. 경험상 답변 품질에 대한 영향은 작지만 이 notebook에서는 측정하지 않으므로 자신의 작업에서 확인하세요.
- **budget을 사용하세요**, latency 목표가 있고 팀이 그에 맞춰 속도를 조절하게 하고 싶을 때입니다.
- **자신의 작업에서는** `TeamClock`, `clock_line`, `stamp_elapsed`, `TIME_MATTERS`를 유지하고 자신의 도구, system prompt, 질문을 제공하세요. 철저함과 속도가 서로 충돌하는 작업을 선택하세요. 절충할 것이 없는 작업은 모든 조건에서 똑같아 보입니다. 여기서 교체할 부분은 `TASK_TOOLS`, `make_task_handlers`, 두 system prompt, 자신의 질문입니다.

budget은 짧은 작업보다 긴 작업에 더 적합합니다. 전체 작업이 1~2분 걸린다면 그 절반의 시간에는 작업을 할 여유가 거의 없으며, Claude는 작업을 포기하는 대신 budget을 넘길 수 있습니다. 여기의 코드를 latency incentive에 관한 자신의 실험을 위한 출발점으로 삼으세요.

더 빠른 조건을 실제로 적용하기 전에 답변이 자신의 작업에 여전히 충분히 좋은지 확인하세요.

자연스러운 다음 단계는 측정입니다. 자신의 작업에서 각 조건을 여러 번 실행하고 실제 경과 시간을 비교하세요. orchestration 자체는 [Async multi-agent orchestration](async_multi_agent_orchestration.ipynb)을 참조하세요.